# 400 · Object-oriented design — procedural layer

**Domain mnemonic — the FOUR pillars:** encapsulation, abstraction, inheritance, polymorphism.

Codes covered: **406** this / self · **413** class invariant · **427** diamond problem · **439** replace switch-on-type with polymorphism · **444** preconditions & postconditions · **451** composition over inheritance · **467** strategy as OCP enabler · **471** rectangle-square problem · **485** equals/hashCode contract · **498** extract method.

**Method:** read the cell → **predict the output** → run it → compare. The matrix holds the imagery; this notebook engraves the net effect. Cells run top to bottom in one kernel. Three cells raise on purpose — their first line says so, and the traceback itself is the output to study.

## 406 · this / self

The implicit reference to the current instance inside a method, used to reach that object's own fields and methods.

*Every method opens by snapping a SELFIE: the first parameter is always a picture of the very object doing the talking.*

Watch: `obj.method()` is nothing but sugar for `Class.method(obj)` — Python always passes the instance as the first argument, and a bound method is just the function pre-loaded with `self`.

In [1]:
class Greeter:
    def __init__(self, name):
        self.name = name

    def hello(self):
        return "Hello, I'm " + self.name

g = Greeter("Ada")
print("g.hello()        ->", g.hello())
print("Greeter.hello(g) ->", Greeter.hello(g))
print("same result?     ->", g.hello() == Greeter.hello(g), " # obj.method() IS Class.method(obj)")
print()
print("g.hello without parentheses is a", type(g.hello).__name__, "object:")
print("  its __self__ is g            ->", g.hello.__self__ is g)
print("  its __func__ is Greeter.hello ->", g.hello.__func__ is Greeter.hello)
print("A bound method = plain function + the selfie already glued in as argument one.")

g.hello()        -> Hello, I'm Ada
Greeter.hello(g) -> Hello, I'm Ada
same result?     -> True  # obj.method() IS Class.method(obj)

g.hello without parentheses is a method object:
  its __self__ is g            -> True
  its __func__ is Greeter.hello -> True
A bound method = plain function + the selfie already glued in as argument one.


In [2]:
class Broken:
    def shout():          # forgot self!
        return "HELLO"

print("Called on the CLASS, it is a plain function -> Broken.shout() =", Broken.shout())
print()
b = Broken()
print("But on an INSTANCE, b.shout() would fail, because b.shout() is Broken.shout(b):")
print("Python passes the instance as argument 1 to a function declared with 0 parameters.")
print("It would raise: TypeError: shout() takes 0 positional arguments but 1 was given")
print("Moral: the selfie parameter is not optional decoration — the machinery relies on it.")

Called on the CLASS, it is a plain function -> Broken.shout() = HELLO

But on an INSTANCE, b.shout() would fail, because b.shout() is Broken.shout(b):
Python passes the instance as argument 1 to a function declared with 0 parameters.
It would raise: TypeError: shout() takes 0 positional arguments but 1 was given
Moral: the selfie parameter is not optional decoration — the machinery relies on it.


## 413 · Class invariant

A condition on object state that must hold between all public calls (e.g. `balance >= 0`); every method must restore it before returning.

*A tightrope walker's balance bar: each trick (method) may wobble it mid-air, but he must be dead level again before the crowd looks up.*

Watch: the property setter is the single checkpoint every state change passes through — legal updates sail through, and the one that would leave the object in an illegal state is refused with a `ValueError` before the damage lands.

In [3]:
class BankAccount:
    def __init__(self, opening):
        self._balance = 0
        self.balance = opening        # even birth goes through the checkpoint

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("invariant violated: balance must be >= 0, got " + str(value))
        self._balance = value

acct = BankAccount(100)
print("opened with            ->", acct.balance)
acct.balance += 50                    # deposit: getter, then setter re-checks
print("after +50 deposit      ->", acct.balance)
acct.balance -= 120                   # withdrawal within funds
print("after -120 withdrawal  ->", acct.balance)
print("invariant balance >= 0 held between every public call")

opened with            -> 100
after +50 deposit      -> 150
after -120 withdrawal  -> 30
invariant balance >= 0 held between every public call


In [4]:
# INTENDED ERROR — read the traceback
# Withdrawing 999 from 30 would leave the balance at -969; the setter refuses to
# let the object exist in that state — the tightrope bar never stays tilted.
print("attempting: acct.balance -= 999   (balance is", acct.balance, ")")
acct.balance -= 999

attempting: acct.balance -= 999   (balance is 30 )


ValueError: invariant violated: balance must be >= 0, got -969

## 427 · Diamond problem

In multiple inheritance, inheriting one ancestor through two parents makes it ambiguous which inherited version applies.

*A family-reunion photo shaped like a DIAMOND: one grandma on top, two parents at the sides, one kid at the bottom clutching two conflicting heirloom recipes for the same pie.*

Watch: with cooperative `super().__init__()`, D's constructor visits **every** class exactly once, in MRO order — the shared grandparent A runs a single time, not twice. `D.__mro__` is the C3 linearization that turns the ambiguous diamond into one deterministic queue.

In [5]:
class A:
    def __init__(self):
        print("A.__init__   (shared grandparent — note it runs exactly ONCE)")

class B(A):
    def __init__(self):
        print("B.__init__   -> handing off via super()")
        super().__init__()

class C(A):
    def __init__(self):
        print("C.__init__   -> handing off via super()")
        super().__init__()

class D(B, C):
    def __init__(self):
        print("D.__init__   -> handing off via super()")
        super().__init__()

print("Constructing D() — predict the order of the four lines:")
d = D()
print()
print("C3 linearization (MRO) — the deterministic answer to 'whose method wins?':")
print("  " + " -> ".join(cls.__name__ for cls in D.__mro__))

Constructing D() — predict the order of the four lines:
D.__init__   -> handing off via super()
B.__init__   -> handing off via super()
C.__init__   -> handing off via super()
A.__init__   (shared grandparent — note it runs exactly ONCE)

C3 linearization (MRO) — the deterministic answer to 'whose method wins?':
  D -> B -> C -> A -> object


## 439 · Replace switch-on-type with polymorphism

Refactoring: turn if/switch chains on type tags into an overridden method, so adding a type requires no edits to existing branches.

*A hallway of doors, each with its own doorman, replaces the lobby clerk flipping a giant laminated flowchart ('if wolf... elif duck...'). New guest? Just add a door.*

Watch: both approaches agree while the type list is frozen. The second cell adds one new class — the polymorphic dispatcher needs **zero** changes, while the isinstance chain falls straight into its else branch.

In [6]:
class Dog:
    def speak(self):
        return "Woof"

class Cat:
    def speak(self):
        return "Meow"

# Approach 1 — the lobby clerk's flowchart: a switch on type tags
def speak_switch(animal):
    if isinstance(animal, Dog):
        return "Woof"
    elif isinstance(animal, Cat):
        return "Meow"
    else:
        return "??? (speak_switch must be EDITED for every new type)"

# Approach 2 — each class owns its door: polymorphic dispatch
def speak_poly(animal):
    return animal.speak()

zoo = [Dog(), Cat()]
for a in zoo:
    print(type(a).__name__.ljust(5), "| switch:", speak_switch(a).ljust(4), "| poly:", speak_poly(a))

Dog   | switch: Woof | poly: Woof
Cat   | switch: Meow | poly: Meow


In [7]:
# A NEW type arrives. Count the edits each approach demands.
class Duck:
    def speak(self):
        return "Quack"      # the new door brings its own doorman

zoo.append(Duck())
print("speak_poly needed ZERO changes; speak_switch is now wrong for Duck:")
for a in zoo:
    print(type(a).__name__.ljust(5), "| switch:", speak_switch(a).ljust(4), "| poly:", speak_poly(a))
print()
print("The branches did not disappear — they moved into the types, where new code lives.")

speak_poly needed ZERO changes; speak_switch is now wrong for Duck:
Dog   | switch: Woof | poly: Woof
Cat   | switch: Meow | poly: Meow
Duck  | switch: ??? (speak_switch must be EDITED for every new type) | poly: Quack

The branches did not disappear — they moved into the types, where new code lives.


## 444 · Preconditions & postconditions

Precondition: what must be true before a call (caller's duty). Postcondition: what the method guarantees afterward (callee's duty).

*Airport rules: YOU must show up with a valid boarding pass (pre); the AIRLINE must land your suitcase on the belt (post). Each side owns its clause.*

Watch: the `@require` decorator checks the caller's clause at the door; the `assert` inside the function checks the callee's clause on the way out. When the error cell fires, read WHO broke the contract — the traceback blames the caller's argument, not the function's logic.

In [8]:
import functools

def require(check, message):
    """Precondition decorator: the boarding-pass check at the gate (caller's duty)."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            assert check(*args, **kwargs), "precondition violated: " + message
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@require(lambda x: x >= 0, "x must be >= 0")
def integer_sqrt(x):
    r = int(x ** 0.5)
    # postcondition — the airline's duty: r is the floor of the true square root
    assert r * r <= x < (r + 1) * (r + 1), "postcondition violated"
    return r

print("integer_sqrt(49) ->", integer_sqrt(49))
print("integer_sqrt(50) ->", integer_sqrt(50))
print("pre held at the gate (caller's duty); post held on landing (callee's duty)")

integer_sqrt(49) -> 7
integer_sqrt(50) -> 7
pre held at the gate (caller's duty); post held on landing (callee's duty)


In [9]:
# INTENDED ERROR — read the traceback
# The CALLER breaks the contract: -9 has no boarding pass for x >= 0.
# The AssertionError fires in the wrapper, BEFORE integer_sqrt's body ever runs.
print("attempting: integer_sqrt(-9)")
integer_sqrt(-9)

attempting: integer_sqrt(-9)


AssertionError: precondition violated: x must be >= 0

## 451 · Composition over inheritance

Prefer assembling behavior from contained parts over subclassing: fewer hierarchy traps, and parts stay swappable at runtime.

*Two robot builders: one WELDS his robot onto grandpa's chassis forever (inheritance); the other clicks modules on and off in seconds and wins the derby. Buy LEGO, not welds.*

Watch: `Car` HAS-A engine, so changing behavior is one attribute assignment on a live object — no `PetrolCar` / `ElectricCar` subclasses anywhere, and the same car object hums after it vroomed.

In [10]:
class PetrolEngine:
    def start(self):
        return "vroom (petrol combustion)"

class ElectricEngine:
    def start(self):
        return "hummm (electric motor)"

class Car:
    def __init__(self, engine):
        self.engine = engine              # HAS-A: a clicked-on module, not a weld

    def drive(self):
        return "Car driving: " + self.engine.start()

car = Car(PetrolEngine())
print(car.drive())

car.engine = ElectricEngine()             # swap the part at RUNTIME
print(car.drive())
print()
print("Same Car object, same Car class, new behavior — zero subclassing.")
print("An inherited PetrolCar would have needed a rebuild; the module just clicked off.")

Car driving: vroom (petrol combustion)
Car driving: hummm (electric motor)

Same Car object, same Car class, new behavior — zero subclassing.
An inherited PetrolCar would have needed a rebuild; the module just clicked off.


## 467 · Strategy as OCP enabler

Encapsulate interchangeable algorithms behind one interface and inject the choice, so new variants extend without editing callers.

*A chess coach's binder of laminated STRATEGY cards: slide 'Sicilian' out, slide 'King's Gambit' in — the player (context) plays whatever card sits in the sleeve, never rewired.*

Watch: `price()` and the existing strategies are never touched. The second cell extends the system by sliding one new card into the binder — a dict entry — and everything downstream just works: open for extension, closed for modification.

In [11]:
def flat_rate(total):
    return round(total + 5.00, 2)         # fixed handling fee

def by_weight(total):
    return round(total * 1.10, 2)         # +10% freight surcharge

PRICING = {
    "flat": flat_rate,
    "weight": by_weight,
}

def price(total, strategy):
    return PRICING[strategy](total)       # the context: plays whatever card is in the sleeve

for name in ["flat", "weight"]:
    print(name.rjust(8), ": 100.00 ->", format(price(100.00, name), ".2f"))

    flat : 100.00 -> 105.00
  weight : 100.00 -> 110.00


In [12]:
# EXTENSION, not modification: one new card slides into the binder.
# price(), flat_rate, by_weight — all untouched.
def member_discount(total):
    return round(total * 0.85, 2)         # loyal customers pay 85%

PRICING["member"] = member_discount

for name in ["flat", "weight", "member"]:
    print(name.rjust(8), ": 100.00 ->", format(price(100.00, name), ".2f"))
print()
print("New behavior arrived as pure addition — the Open/Closed Principle in one dict entry.")

    flat : 100.00 -> 105.00
  weight : 100.00 -> 110.00
  member : 100.00 -> 85.00

New behavior arrived as pure addition — the Open/Closed Principle in one dict entry.


## 471 · Rectangle-square problem

Classic LSP violation: a Square subclassing Rectangle breaks code that sets width and height independently.

*A stretching machine grabs a 'rectangle' by both handles; the square SCREAMS as both its sides move together, crushing the operator's test gauge.*

Watch: the client function was written against Rectangle's contract — width and height move independently. Square keeps its sides locked together, so `height = 5` silently overwrites `width = 4`, and 20 becomes 25 with no exception anywhere. That silence is the whole danger.

In [13]:
class Rectangle:
    def __init__(self):
        self._w = 0
        self._h = 0

    @property
    def width(self):
        return self._w

    @width.setter
    def width(self, v):
        self._w = v

    @property
    def height(self):
        return self._h

    @height.setter
    def height(self, v):
        self._h = v

    def area(self):
        return self._w * self._h

class Square(Rectangle):
    """Mathematically IS-A rectangle — behaviorally, the sides are welded together."""
    @Rectangle.width.setter
    def width(self, v):
        self._w = v
        self._h = v

    @Rectangle.height.setter
    def height(self, v):
        self._w = v
        self._h = v

def stretch_to_4x5(shape):
    """Client code written against Rectangle's contract: the two sides are independent."""
    shape.width = 4
    shape.height = 5
    expected, got = 20, shape.area()
    verdict = "OK" if got == expected else "LSP VIOLATED"
    print(type(shape).__name__.rjust(9), ": expected area", expected, "| got", got, "->", verdict)

stretch_to_4x5(Rectangle())
stretch_to_4x5(Square())
print()
print("No exception was raised — the subclass broke the client SILENTLY.")
print("Substitutability is about behavior, not about what the type checker accepts.")

Rectangle : expected area 20 | got 20 -> OK
   Square : expected area 20 | got 25 -> LSP VIOLATED

No exception was raised — the subclass broke the client SILENTLY.
Substitutability is about behavior, not about what the type checker accepts.


## 485 · equals/hashCode contract

If two objects are equal they MUST have equal hash codes; violating this silently breaks hash sets and dict lookups.

*A coat check: the clerk files coats by ticket number (hash), then confirms by face (equals). Twins holding different ticket numbers means one coat is lost in the wrong rack forever.*

Watch: Python enforces half the contract for you — defining `__eq__` alone sets `__hash__` to `None`, making instances unhashable (the error cell). The fix is a `__hash__` built from the **same fields** as `__eq__`, so equal twins always hold equal ticket numbers.

In [14]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)

p, q = Point(1, 2), Point(1, 2)
print("p == q ?               ", p == q)
print("Point.__hash__ is None?", Point.__hash__ is None, "  # defining __eq__ alone DISABLES hashing")
print("Equal faces, but no ticket numbers — these Points cannot enter a set or key a dict.")

p == q ?                True
Point.__hash__ is None? True   # defining __eq__ alone DISABLES hashing
Equal faces, but no ticket numbers — these Points cannot enter a set or key a dict.


In [15]:
# INTENDED ERROR — read the traceback
# __eq__ without __hash__ makes instances unhashable: the coat check cannot
# file a coat that has a face but no ticket number.
print("attempting: {Point(1, 2)}")
s = {Point(1, 2)}

attempting: {Point(1, 2)}


TypeError: unhashable type: 'Point'

In [16]:
class HashablePoint:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return isinstance(other, HashablePoint) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))     # SAME fields as __eq__ -> contract holds

a, b = HashablePoint(1, 2), HashablePoint(1, 2)
print("a == b ?              ", a == b)
print("hash(a) == hash(b) ?  ", hash(a) == hash(b), "  # equal objects, equal tickets")
coats = {a}
print("b in {a} ?            ", b in coats, "  # the set finds the equal twin")
labels = {a: "checked in"}
print('labels[b]              ->', repr(labels[b]), "  # dict lookup by an equal key works")

a == b ?               True
hash(a) == hash(b) ?   True   # equal objects, equal tickets
b in {a} ?             True   # the set finds the equal twin
labels[b]              -> 'checked in'   # dict lookup by an equal key works


## 498 · Extract method

Refactoring: pull a coherent code fragment into a new well-named method and call it, turning comments into names.

*A dentist EXTRACTS one tangled tooth from the jaw (long method) and mounts it on its own little stand with a nameplate — the jaw finally closes, and the tooth can be inspected alone.*

Watch: the `# -- section --` comments in v1 are begging to be names. v2 rebuilds the exact same report from three named helpers, and the final `assert` proves the refactor changed the shape of the code without changing its behavior.

In [17]:
def report_v1(orders):
    # one long function: header, body and footer all tangled in a single jaw
    lines = []
    # -- format header --
    lines.append("=" * 30)
    lines.append("SALES REPORT".center(30))
    lines.append("=" * 30)
    # -- format body --
    total = 0
    for name, qty, price in orders:
        amount = qty * price
        total += amount
        lines.append(name.ljust(10) + str(qty).rjust(3) + " x " + format(price, "6.2f") + " = " + format(amount, "7.2f"))
    # -- format footer --
    lines.append("-" * 30)
    lines.append("TOTAL".ljust(10) + format(total, "20.2f"))
    return "\n".join(lines)

orders = [("widget", 3, 2.50), ("gizmo", 1, 9.99), ("doodad", 10, 0.75)]
print(report_v1(orders))

         SALES REPORT         
widget      3 x   2.50 =    7.50
gizmo       1 x   9.99 =    9.99
doodad     10 x   0.75 =    7.50
------------------------------
TOTAL                    24.99


In [18]:
# Each commented block becomes a NAMED helper: the comment dies, the name survives.
def header(title, width=30):
    return ["=" * width, title.center(width), "=" * width]

def body(orders):
    lines, total = [], 0
    for name, qty, price in orders:
        amount = qty * price
        total += amount
        lines.append(name.ljust(10) + str(qty).rjust(3) + " x " + format(price, "6.2f") + " = " + format(amount, "7.2f"))
    return lines, total

def footer(total, width=30):
    return ["-" * width, "TOTAL".ljust(10) + format(total, "20.2f")]

def report_v2(orders):
    body_lines, total = body(orders)
    return "\n".join(header("SALES REPORT") + body_lines + footer(total))

print(report_v2(orders))
print()
assert report_v2(orders) == report_v1(orders)
print("assert passed: byte-identical output — only the SHAPE of the code changed.")

         SALES REPORT         
widget      3 x   2.50 =    7.50
gizmo       1 x   9.99 =    9.99
doodad     10 x   0.75 =    7.50
------------------------------
TOTAL                    24.99

assert passed: byte-identical output — only the SHAPE of the code changed.
